# FlowGraphFormer — 04_ROC_Curves.ipynb

Loads each of the 7 **saved final models** from `models/trained/` (produced by
`03_Big_Modeling.ipynb`) and, for every model, plots **one figure with two ROC
curves** — Validation and Test — so you can see generalization gap per model
at a glance.

**Does not retrain anything.** Only loads saved weights and runs inference on
`X_val` / `X_test` (loaded fresh from `01_Preprocessing_local.ipynb`'s output, exactly
like `03_Big_Modeling.ipynb` Cell 5).

Toggle `TASK_MODE` in the Master Control Cell:
- `"binary"` — BENIGN vs ATTACK (collapses all attack classes into one positive class), single ROC curve.
- `"multiclass"` — macro-average One-vs-Rest ROC curve (interpolated, sklearn-standard recipe), across all classes.

Depends only on artifacts already produced by `01_Preprocessing_local.ipynb` and `03_Big_Modeling.ipynb`
(processed parquet/csv splits, `label_encoder.pkl`, and `models/trained/*`).


In [ ]:
# ==============================================================================
# Cell 1 : Mount Drive & Project Setup
# ==============================================================================
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_ENVIRONMENT = "colab"
    PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project/FlowGraphFormer")
except ModuleNotFoundError:
    RUN_ENVIRONMENT = "local"
    PROJECT_ROOT = Path.cwd()
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print("FlowGraphFormer")
print("Research Notebook : 04_ROC_Curves")
print("=" * 80)
print("Run Environment :", RUN_ENVIRONMENT)
print("Project Root    :", PROJECT_ROOT)

import gc

def print_ram_usage(label=""):
    try:
        import psutil, os as _os
        process = psutil.Process(_os.getpid())
        rss_gb = process.memory_info().rss / 1024 ** 3
        total_gb = psutil.virtual_memory().total / 1024 ** 3
        print(f"RAM Usage - {label} : {rss_gb:.2f} GB / {total_gb:.2f} GB")
    except ImportError:
        pass

def downcast_dtypes(df, verbose=True):
    import pandas as pd
    start_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    end_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    if verbose:
        print(f"Downcast dtypes : {start_mb:,.1f} MB -> {end_mb:,.1f} MB "
              f"({(1 - end_mb / start_mb) * 100:.1f}% reduction)")
    return df

def cleanup_torch():
    """Frees Python-side RAM and, if a GPU is in use, releases its cached
    memory pool. Handles NVIDIA/AMD (cuda), Intel (xpu), and Apple (mps)."""
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        if hasattr(torch, "xpu") and torch.xpu.is_available():
            torch.xpu.empty_cache()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
    except ImportError:
        pass
    except Exception:
        pass


In [ ]:
# ==============================================================================
# Cell 2 : Install / Verify Required Packages
# ==============================================================================
import sys, subprocess, importlib

REQUIRED_PACKAGES = {
    "catboost": "catboost",
    "torch": "torch",
    "tabpfn": "tabpfn",
}

missing_packages = []
for module, package in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print("Installing :", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing_packages)
else:
    print("All required packages already installed.")


In [ ]:
# ==============================================================================
# Cell 3 : Project Directories (mirrors 03_Big_Modeling.ipynb Cell 3)
# ==============================================================================
from pathlib import Path

DATASET_DIR = PROJECT_ROOT / "dataset"
PROCESSED_DATASET_DIR = DATASET_DIR / "processed" / "final"

REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_REPORT_DIR = REPORT_DIR / "modeling"
FIGURE_DIR = MODEL_REPORT_DIR / "figures" / "roc_avg"   # separate subfolder, won't clobber 03's figures
TABLE_DIR = MODEL_REPORT_DIR / "tables"

MODEL_DIR = PROJECT_ROOT / "models"
TRAINED_MODEL_DIR = MODEL_DIR / "trained"
ENCODER_DIR = MODEL_DIR / "encoders"

for d in [FIGURE_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROCESSED_DATASET_DIR :", PROCESSED_DATASET_DIR)
print("TRAINED_MODEL_DIR     :", TRAINED_MODEL_DIR)
print("FIGURE_DIR (this nb)  :", FIGURE_DIR)


## Master Control Cell

Everything important is controlled here. `TASK_MODE` is the main toggle:
run the cell, change it, re-run from here down — no need to touch other cells.


In [ ]:
# ==============================================================================
# Cell 4 : MASTER CONTROL CELL
# ==============================================================================

RANDOM_SEED = 42  #@param {type:"integer"}

# ------------------------------------------------------------------
# Dataset files -- MUST match what 03_Big_Modeling.ipynb Cell 4 used to
# train the models you are about to load, or the feature columns / scaling
# won't line up with what each model was trained on.
# ------------------------------------------------------------------
VAL_FEATURES_FILE  = "X_validation_scaled.parquet"  #@param {type:"string"}
TEST_FEATURES_FILE = "X_test_scaled.parquet"        #@param {type:"string"}
VAL_LABELS_FILE    = "y_validation.csv"             #@param {type:"string"}
TEST_LABELS_FILE   = "y_test.csv"                   #@param {type:"string"}

# ------------------------------------------------------------------
# Task mode
# "binary"      -> collapses every non-BENIGN class into a single ATTACK
#                  class (1). Positive class score = 1 - P(BENIGN), i.e.
#                  the total probability mass placed on any attack class.
# "multiclass"  -> macro-average One-vs-Rest ROC curve across all classes
#                  (standard sklearn macro-average recipe: interpolate all
#                  per-class ROC curves onto one FPR grid, then average).
# ------------------------------------------------------------------
TASK_MODE = "multiclass"   #@param ["binary", "multiclass"]

BENIGN_CLASS_NAME = "BENIGN"  #@param {type:"string"}

# ------------------------------------------------------------------
# Compute backend for the 5 PyTorch models + TabPFN (CatBoost reads its
# own saved task_type from the .cbm file, no device choice needed here)
# ------------------------------------------------------------------
DEVICE_PREFERENCE = "auto"   #@param ["auto", "cuda", "xpu", "mps", "cpu"]
GPU_INDEX = 0                 #@param {type:"integer"}

# Inference batch size for the PyTorch models -- kept modest by default
# since TabGT / GNNFormer / GraphTransformer build O(batch^2) attention
# matrices; lower this if you hit an OOM on val/test inference.
INFERENCE_BATCH_SIZE = 2048   #@param {type:"integer"}

# TabPFN was trained on a subsampled / column-truncated view of the data
# (see 03_Big_Modeling.ipynb Cell 16) -- these must match what it was
# trained with, or predict_proba will error on a feature-count mismatch.
TABPFN_MAX_FEATURES = 100     #@param {type:"integer"}

# ------------------------------------------------------------------
# Which of the 7 models to attempt. Any model whose saved file is
# missing is skipped with a warning rather than crashing the whole run.
# ------------------------------------------------------------------
MODELS_TO_RUN = [
    "catboost", "fttransformer", "tabm", "tabgt",
    "gnnformer", "graphtransformer", "tabpfn",
]

# ------------------------------------------------------------------
# Hybrid / Stacked Ensemble -- mirrors 03_Big_Modeling.ipynb Cell 17.
# Set INCLUDE_HYBRID=False to skip it entirely.
# HYBRID_MEMBERS must be the SAME set of member models that
# 03_Big_Modeling.ipynb used to build the hybrid, since:
#   - "soft_voting" -> averages each member's val/test proba, computed
#                      fresh here from the already-loaded base models
#                      (no extra file needed).
#   - "stacking"    -> loads the saved meta-model file, whose filename is
#                      f"hybrid_{\'_\'.join(HYBRID_MEMBERS)}_meta_model.pkl"
#                      -- must match exactly (same members) or it won't be found.
# ------------------------------------------------------------------
INCLUDE_HYBRID = True   #@param {type:"boolean"}

HYBRID_STRATEGY = "stacking"   #@param ["stacking", "soft_voting"]

HYBRID_MEMBERS = ["gnnformer", "fttransformer", "graphtransformer"]  #@param {type:"raw"}

FIGURE_DPI = 300      #@param {type:"integer"}
FIGURE_FORMAT = "png" #@param ["png", "pdf", "svg"]

CONFIG = {
    "random_seed": RANDOM_SEED,
    "task_mode": TASK_MODE,
    "benign_class_name": BENIGN_CLASS_NAME,
    "device_preference": DEVICE_PREFERENCE,
    "gpu_index": GPU_INDEX,
    "inference_batch_size": INFERENCE_BATCH_SIZE,
    "tabpfn_max_features": TABPFN_MAX_FEATURES,
    "figure_dpi": FIGURE_DPI,
    "figure_format": FIGURE_FORMAT,
    "include_hybrid": INCLUDE_HYBRID,
    "hybrid_strategy": HYBRID_STRATEGY,
    "hybrid_members": HYBRID_MEMBERS,
}

import random
import numpy as np
random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

try:
    import torch
    torch.manual_seed(CONFIG["random_seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(CONFIG["random_seed"])
except ImportError:
    pass

print("TASK_MODE :", CONFIG["task_mode"])
print("MODELS_TO_RUN :", MODELS_TO_RUN)
if CONFIG["include_hybrid"]:
    print("HYBRID     : strategy=" + CONFIG["hybrid_strategy"] + ", members=" + str(CONFIG["hybrid_members"]))


## Load Validation + Test Data (and label encoder)


In [ ]:
# ==============================================================================
# Cell 5 : Load Validation + Test Splits Only
# ==============================================================================
# Deliberately does NOT load X_train / X_train_smote -- this notebook only
# runs inference, so the (much larger) SMOTE-balanced training file is
# skipped entirely to save RAM.
# ==============================================================================
import pandas as pd
import numpy as np
import joblib
from tqdm.auto import tqdm

print("=" * 80)
print("Loading Validation + Test Data")
print("=" * 80)

FILES_TO_LOAD = [
    ("X_val",  PROCESSED_DATASET_DIR / VAL_FEATURES_FILE),
    ("X_test", PROCESSED_DATASET_DIR / TEST_FEATURES_FILE),
    ("y_val",  PROCESSED_DATASET_DIR / VAL_LABELS_FILE),
    ("y_test", PROCESSED_DATASET_DIR / TEST_LABELS_FILE),
]

loaded = {}
for name, filepath in tqdm(FILES_TO_LOAD, desc="Loading Files"):
    print(f"\nLoading : {name:<8} <- {filepath.name}")
    if not filepath.exists():
        raise FileNotFoundError(f"Missing file expected at: {filepath}")
    if filepath.suffix == ".parquet":
        df = pd.read_parquet(filepath, engine="pyarrow")
    else:
        df = pd.read_csv(filepath)
    downcast_dtypes(df, verbose=False)
    loaded[name] = df
    print(f"  Shape  : {df.shape}")

X_val  = loaded["X_val"]
X_test = loaded["X_test"]
y_val  = loaded["y_val"].iloc[:, 0].values
y_test = loaded["y_test"].iloc[:, 0].values

print(f"\nLoading : label_encoder <- {ENCODER_DIR / 'label_encoder.pkl'}")
label_encoder = joblib.load(ENCODER_DIR / "label_encoder.pkl")

CLASS_NAMES = list(label_encoder.classes_)
N_CLASSES = len(CLASS_NAMES)
FEATURE_NAMES = list(X_val.columns)
N_FEATURES = len(FEATURE_NAMES)

assert list(X_val.columns) == list(X_test.columns), \
    "X_val / X_test columns don't match -- check VAL_FEATURES_FILE / TEST_FEATURES_FILE."

print()
print(f"N_FEATURES : {N_FEATURES}")
print(f"N_CLASSES  : {N_CLASSES}  -> {CLASS_NAMES}")
print(f"X_val  : {X_val.shape}   y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}   y_test : {y_test.shape}")

if CONFIG["benign_class_name"] not in CLASS_NAMES:
    raise ValueError(
        f'BENIGN_CLASS_NAME="{CONFIG["benign_class_name"]}" not found in '
        f"label_encoder classes: {CLASS_NAMES}. Fix BENIGN_CLASS_NAME in Cell 4."
    )

BENIGN_CLASS_IDX = CLASS_NAMES.index(CONFIG["benign_class_name"])
print(f"BENIGN class index : {BENIGN_CLASS_IDX}")


## Device Helper (same cross-vendor logic as 03_Big_Modeling.ipynb)


In [ ]:
# ==============================================================================
# Cell 6 : get_device()
# ==============================================================================
def get_device():
    import torch
    pref = CONFIG["device_preference"]
    idx = CONFIG.get("gpu_index", 0)

    def _cuda_ok():
        try:
            return torch.cuda.is_available() and torch.cuda.device_count() > idx
        except Exception:
            return False

    def _xpu_ok():
        try:
            if not hasattr(torch, "xpu"):
                import intel_extension_for_pytorch  # noqa: F401
            return hasattr(torch, "xpu") and torch.xpu.is_available() and torch.xpu.device_count() > idx
        except Exception:
            return False

    def _mps_ok():
        try:
            return torch.backends.mps.is_available()
        except Exception:
            return False

    if pref == "cpu":
        return torch.device("cpu")
    if pref == "cuda":
        if _cuda_ok():
            return torch.device(f"cuda:{idx}")
        print(f'  WARNING: DEVICE_PREFERENCE="cuda" but no GPU at index {idx} found -- falling back to CPU.')
        return torch.device("cpu")
    if pref == "xpu":
        if _xpu_ok():
            return torch.device(f"xpu:{idx}")
        print(f'  WARNING: DEVICE_PREFERENCE="xpu" but no Intel GPU at index {idx} found -- falling back to CPU.')
        return torch.device("cpu")
    if pref == "mps":
        if _mps_ok():
            return torch.device("mps")
        print('  WARNING: DEVICE_PREFERENCE="mps" but Apple MPS is not available -- falling back to CPU.')
        return torch.device("cpu")

    if _cuda_ok():
        return torch.device(f"cuda:{idx}")
    if _xpu_ok():
        return torch.device(f"xpu:{idx}")
    if _mps_ok():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
print("Resolved device :", DEVICE)


## Model Architectures

Exact copies of the class definitions from `03_Big_Modeling.ipynb` (default
hyperparameters, since every model there was constructed as
`ModelClass(N_FEATURES, N_CLASSES)`). Needed to reconstruct the module before
`load_state_dict()`.


In [ ]:
# ==============================================================================
# Cell 7 : PyTorch Model Architectures (must match 03_Big_Modeling.ipynb exactly)
# ==============================================================================
import torch
import torch.nn as nn


class FTTransformer(nn.Module):
    def __init__(self, n_features, n_classes, d_token=32, n_layers=3, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.feature_weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.feature_bias = nn.Parameter(torch.zeros(n_features, d_token))
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_token) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads, dim_feedforward=d_token * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, n_classes)

    def forward(self, x):
        x = torch.clamp(x, -10.0, 10.0)
        tokens = x.unsqueeze(-1) * self.feature_weight + self.feature_bias
        cls = self.cls_token.expand(x.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = self.norm(encoded[:, 0])
        return self.head(cls_out)


class TabM(nn.Module):
    def __init__(self, n_features, n_classes, hidden_dim=128, k_members=8, dropout=0.1):
        super().__init__()
        self.k = k_members
        self.body = nn.Sequential(
            nn.Linear(n_features, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        self.member_scale = nn.Parameter(torch.randn(k_members, hidden_dim) * 0.02 + 1.0)
        self.member_bias = nn.Parameter(torch.zeros(k_members, hidden_dim))
        self.shared_head = nn.Linear(hidden_dim, n_classes)
        self.member_head_scale = nn.Parameter(torch.randn(k_members, n_classes) * 0.02 + 1.0)

    def forward(self, x):
        shared = self.body(x)
        member_feats = shared.unsqueeze(1) * self.member_scale.unsqueeze(0) + self.member_bias.unsqueeze(0)
        member_logits = self.shared_head(member_feats) * self.member_head_scale.unsqueeze(0)
        return member_logits.mean(dim=1)


class TabGT(nn.Module):
    def __init__(self, n_features, n_classes, d_model=64, k_neighbors=8, n_heads=4, dropout=0.1):
        super().__init__()
        self.k = k_neighbors
        self.input_proj = nn.Linear(n_features, d_model)
        self.neighbor_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Linear(d_model * 2, d_model))
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.input_proj(x)
        with torch.no_grad():
            dist = torch.cdist(h, h)
            k = max(min(self.k, h.size(0) - 1), 0)
            _, nn_idx = torch.topk(-dist, k=k + 1, dim=1)
        neighbor_feats = h[nn_idx]
        attn_out, _ = self.neighbor_attn(h.unsqueeze(1), neighbor_feats, neighbor_feats)
        h = self.norm1(h + attn_out.squeeze(1))
        self_out, _ = self.self_attn(h.unsqueeze(1), h.unsqueeze(1), h.unsqueeze(1))
        h = self.norm2(h + self_out.squeeze(1))
        h = h + self.ffn(h)
        return self.head(h)


class GCNLayer(nn.Module):
    def __init__(self, n_nodes, d_model):
        super().__init__()
        self.adj_logits = nn.Parameter(torch.randn(n_nodes, n_nodes) * 0.01)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, node_feats):
        adj = torch.softmax(self.adj_logits + self.adj_logits.t(), dim=-1)
        aggregated = torch.einsum("ij,bjd->bid", adj, node_feats)
        return torch.relu(self.proj(aggregated) + node_feats)


class GNNFormer(nn.Module):
    def __init__(self, n_features, n_classes, d_model=32, n_gcn_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.gcn_layers = nn.ModuleList([GCNLayer(n_features, d_model) for _ in range(n_gcn_layers)])
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.former = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.pool_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = torch.clamp(x, -10.0, 10.0)
        node_feats = x.unsqueeze(-1) * self.token_embed.unsqueeze(0)
        for gcn in self.gcn_layers:
            node_feats = gcn(node_feats)
        encoded = self.former(node_feats)
        pooled = self.pool_norm(encoded.mean(dim=1))
        return self.head(pooled)


class GraphTransformer(nn.Module):
    def __init__(self, n_features, n_classes, d_model=32, n_layers=3, n_heads=4, dropout=0.1):
        super().__init__()
        self.token_embed = nn.Parameter(torch.randn(n_features, d_model) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.graph_attn_layers = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.readout_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = torch.clamp(x, -10.0, 10.0)
        node_feats = x.unsqueeze(-1) * self.token_embed.unsqueeze(0) + self.pos_embed
        encoded = self.graph_attn_layers(node_feats)
        pooled = self.readout_norm(encoded.mean(dim=1))
        return self.head(pooled)


TORCH_MODEL_CLASSES = {
    "fttransformer": FTTransformer,
    "tabm": TabM,
    "tabgt": TabGT,
    "gnnformer": GNNFormer,
    "graphtransformer": GraphTransformer,
}

print("Model classes ready:", list(TORCH_MODEL_CLASSES.keys()))


## Model Loaders + Chunked Inference


In [ ]:
# ==============================================================================
# Cell 8 : Load-Model + Predict-Proba Helpers
# ==============================================================================
# Chunked inference mirrors 03_Big_Modeling.ipynb's train_torch_classifier()
# eval loop: TabGT / GNNFormer / GraphTransformer build O(batch^2) attention
# / distance matrices, so predicting on the full val/test set in one forward
# pass risks a CUDA OOM. We predict in CONFIG["inference_batch_size"] chunks
# instead, which changes nothing about the math (batch-independent forward
# passes) but keeps memory bounded.
# ==============================================================================
from catboost import CatBoostClassifier


def load_catboost():
    path = TRAINED_MODEL_DIR / "catboost_final.cbm"
    if not path.exists():
        raise FileNotFoundError(path)
    model = CatBoostClassifier()
    model.load_model(str(path))
    return model


def catboost_predict_proba(model, X):
    return model.predict_proba(X)


def load_torch_model(model_key):
    path = TRAINED_MODEL_DIR / f"{model_key}_final.pt"
    if not path.exists():
        raise FileNotFoundError(path)
    cls = TORCH_MODEL_CLASSES[model_key]
    model = cls(N_FEATURES, N_CLASSES).to(DEVICE)
    state_dict = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    return model


def torch_predict_proba(model, X):
    """Chunked softmax inference. X is a DataFrame or ndarray."""
    X_vals = X.values if hasattr(X, "values") else X
    X_t = torch.tensor(X_vals, dtype=torch.float32)
    batch_size = CONFIG["inference_batch_size"]
    all_proba = []
    model.eval()
    with torch.no_grad():
        for start in range(0, X_t.size(0), batch_size):
            chunk = X_t[start:start + batch_size].to(DEVICE)
            logits = model(chunk)
            if torch.isnan(logits).any() or torch.isinf(logits).any():
                print("  WARNING: NaN/Inf logits during inference -- replacing with zeros.")
                logits = torch.zeros_like(logits)
            proba = torch.softmax(logits, dim=1).cpu().numpy()
            all_proba.append(proba)
    return np.concatenate(all_proba, axis=0)


def load_tabpfn():
    path = TRAINED_MODEL_DIR / "tabpfn_final.pkl"
    if not path.exists():
        raise FileNotFoundError(path)
    try:
        return joblib.load(path)
    except RuntimeError as exc:
        # PyTorch storage embedded in a joblib pickle remembers its original
        # CUDA device. Retry with an explicit map_location on CPU-only hosts.
        if "deserialize object on a CUDA device" not in str(exc):
            raise
        import io
        original_loader = torch.storage._load_from_bytes
        torch.storage._load_from_bytes = lambda data: torch.load(
            io.BytesIO(data), map_location=DEVICE, weights_only=False
        )
        try:
            return joblib.load(path)
        finally:
            torch.storage._load_from_bytes = original_loader


def tabpfn_predict_proba(model, X):
    # TabPFN was trained on at most CONFIG["tabpfn_max_features"] columns
    # (see 03_Big_Modeling.ipynb Cell 16) -- truncate the same way here.
    n_cols = min(X.shape[1], CONFIG["tabpfn_max_features"])
    X_sub = X.iloc[:, :n_cols]
    return model.predict_proba(X_sub)


# Registry: model_key -> (loader_fn, predict_proba_fn)
MODEL_REGISTRY = {
    "catboost":         (load_catboost, catboost_predict_proba),
    "fttransformer":    (lambda: load_torch_model("fttransformer"), torch_predict_proba),
    "tabm":             (lambda: load_torch_model("tabm"), torch_predict_proba),
    "tabgt":            (lambda: load_torch_model("tabgt"), torch_predict_proba),
    "gnnformer":        (lambda: load_torch_model("gnnformer"), torch_predict_proba),
    "graphtransformer": (lambda: load_torch_model("graphtransformer"), torch_predict_proba),
    "tabpfn":           (load_tabpfn, tabpfn_predict_proba),
}

DISPLAY_NAMES = {
    "catboost": "CatBoost",
    "fttransformer": "FT-Transformer",
    "tabm": "TabM",
    "tabgt": "TabGT",
    "gnnformer": "GNNFormer",
    "graphtransformer": "GraphTransformer",
    "tabpfn": "TabPFN-2.5",
}

print("Model registry ready:", list(MODEL_REGISTRY.keys()))


## ROC Computation (binary or macro-average multiclass) + Plotting


In [ ]:
# ==============================================================================
# Cell 9 : ROC Curve Computation + Single-Figure (Val + Test) Plotting
# ==============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc


def to_binary_labels(y):
    """1 = any attack class, 0 = BENIGN."""
    return (y != BENIGN_CLASS_IDX).astype(int)


def to_binary_score(y_proba):
    """P(attack) = 1 - P(BENIGN) = sum of every non-BENIGN class probability."""
    return 1.0 - y_proba[:, BENIGN_CLASS_IDX]


def compute_roc_binary(y_true, y_proba):
    y_bin = to_binary_labels(y_true)
    scores = to_binary_score(y_proba)
    fpr, tpr, _ = roc_curve(y_bin, scores)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc


def compute_roc_macro_ovr(y_true, y_proba):
    """Standard sklearn macro-average multiclass ROC recipe:
    compute each class's OvR ROC curve, interpolate all of them onto one
    shared FPR grid, then average the TPRs. Classes absent from y_true in
    this split are skipped (undefined ROC) with a printed note."""
    fpr_dict, tpr_dict = {}, {}
    valid_classes = []

    for class_idx in range(N_CLASSES):
        y_true_bin = (y_true == class_idx).astype(int)
        if y_true_bin.sum() == 0:
            print(f"    NOTE: class '{CLASS_NAMES[class_idx]}' has 0 samples in this "
                  f"split -- excluded from the macro-average.")
            continue
        fpr, tpr, _ = roc_curve(y_true_bin, y_proba[:, class_idx])
        fpr_dict[class_idx] = fpr
        tpr_dict[class_idx] = tpr
        valid_classes.append(class_idx)

    all_fpr = np.unique(np.concatenate([fpr_dict[c] for c in valid_classes]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in valid_classes:
        mean_tpr += np.interp(all_fpr, fpr_dict[c], tpr_dict[c])
    mean_tpr /= len(valid_classes)

    macro_auc = auc(all_fpr, mean_tpr)
    return all_fpr, mean_tpr, macro_auc


def compute_roc(y_true, y_proba):
    if CONFIG["task_mode"] == "binary":
        return compute_roc_binary(y_true, y_proba)
    return compute_roc_macro_ovr(y_true, y_proba)


def plot_val_test_roc(model_display_name, fpr_val, tpr_val, auc_val, fpr_test, tpr_test, auc_test):
    fig, ax = plt.subplots(figsize=(7, 6.5))

    ax.plot(fpr_val, tpr_val, color="#1f77b4", lw=2.2,
            label=f"Validation (AUC = {auc_val:.4f})")
    ax.plot(fpr_test, tpr_test, color="#d62728", lw=2.2,
            label=f"Test (AUC = {auc_test:.4f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey", lw=1, label="Chance")

    curve_kind = "Binary" if CONFIG["task_mode"] == "binary" else "Macro-Average OvR"
    ax.set_title(f"{model_display_name} — {curve_kind} ROC Curve", fontsize=14)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.legend(loc="lower right")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    out_path = FIGURE_DIR / f"{model_display_name.lower().replace(' ', '_').replace('-', '')}_val_test_roc.{CONFIG['figure_format']}"
    plt.savefig(out_path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
    plt.show()
    return out_path

print("ROC helpers ready: compute_roc, plot_val_test_roc")


## Run — Load Each Saved Model, Predict on Val + Test, Plot

Skips (with a warning, not a crash) any model whose saved file isn't found —
e.g. if you haven't trained/saved a particular model yet in `03_Big_Modeling.ipynb`.


In [ ]:
# ==============================================================================
# Cell 10 : Main Loop — one figure per model
# ==============================================================================
summary_rows = []

# Cache of each base model's val/test probability arrays, keyed by model_key.
# Reused by the Hybrid cell below so it doesn't need to reload/re-infer a
# member model that was already run in this loop.
ALL_VAL_PROBA = {}
ALL_TEST_PROBA = {}

for model_key in MODELS_TO_RUN:

    display_name = DISPLAY_NAMES.get(model_key, model_key)

    print("=" * 80)
    print(f"[{model_key}] Loading saved model + running inference")
    print("=" * 80)

    if model_key not in MODEL_REGISTRY:
        print(f"  SKIPPED: '{model_key}' is not a recognized model key.")
        continue

    loader_fn, predict_fn = MODEL_REGISTRY[model_key]

    try:
        model = loader_fn()
    except FileNotFoundError as e:
        print(f"  SKIPPED: saved model file not found -> {e}")
        summary_rows.append({"model": display_name, "status": "missing_file"})
        continue
    except Exception as e:
        print(f"  SKIPPED: failed to load model -> {type(e).__name__}: {e}")
        summary_rows.append({"model": display_name, "status": f"load_error: {e}"})
        continue

    try:
        val_proba = predict_fn(model, X_val)
        test_proba = predict_fn(model, X_test)

        ALL_VAL_PROBA[model_key] = val_proba
        ALL_TEST_PROBA[model_key] = test_proba

        fpr_val, tpr_val, auc_val = compute_roc(y_val, val_proba)
        fpr_test, tpr_test, auc_test = compute_roc(y_test, test_proba)

        print(f"  Validation AUC : {auc_val:.4f}")
        print(f"  Test AUC       : {auc_test:.4f}")

        out_path = plot_val_test_roc(display_name, fpr_val, tpr_val, auc_val,
                                      fpr_test, tpr_test, auc_test)
        print(f"  Saved figure   : {out_path}")

        summary_rows.append({
            "model": display_name,
            "status": "ok",
            "val_auc": round(auc_val, 4),
            "test_auc": round(auc_test, 4),
            "generalization_gap": round(auc_val - auc_test, 4),
        })

    except Exception as e:
        print(f"  SKIPPED: inference/plotting failed -> {type(e).__name__}: {e}")
        summary_rows.append({"model": display_name, "status": f"inference_error: {e}"})

    finally:
        del model
        cleanup_torch()
        print_ram_usage(f"after {model_key}")

    print()

print("Done.")


## Hybrid Model — Stacked Ensemble ROC

Mirrors `03_Big_Modeling.ipynb`'s hybrid cell: combines `HYBRID_MEMBERS`
(Master Control Cell) via **stacking** (loads the saved logistic-regression
meta-model and feeds it each member's val/test probabilities) or
**soft voting** (simple average of member probabilities — no saved file
needed). Any member not already evaluated in the loop above is loaded and
run here on the fly.

Set `INCLUDE_HYBRID = False` in the Master Control Cell to skip this.


In [ ]:
# ==============================================================================
# Cell 10b : Hybrid / Stacked Ensemble — ROC (Val + Test)
# ==============================================================================
if not CONFIG["include_hybrid"]:

    print("INCLUDE_HYBRID=False -- skipping hybrid model.")

else:

    hybrid_members = CONFIG["hybrid_members"]
    hybrid_strategy = CONFIG["hybrid_strategy"]
    hybrid_display_name = "Hybrid (" + " + ".join(DISPLAY_NAMES.get(m, m) for m in hybrid_members) + ")"

    print("=" * 80)
    print(f"[hybrid] strategy={hybrid_strategy}  members={hybrid_members}")
    print("=" * 80)

    try:
        # ------------------------------------------------------------
        # Ensure every member's val/test proba is available -- reuse
        # from the main loop if already computed, otherwise load+infer
        # fresh (a member can be a hybrid ingredient without being in
        # MODELS_TO_RUN).
        # ------------------------------------------------------------
        for member in hybrid_members:
            if member not in ALL_VAL_PROBA:
                if member not in MODEL_REGISTRY:
                    raise ValueError(f"Unknown hybrid member '{member}' -- not in MODEL_REGISTRY.")
                print(f"  [hybrid] '{member}' wasn't run above -- loading + inferring now.")
                loader_fn, predict_fn = MODEL_REGISTRY[member]
                member_model = loader_fn()
                ALL_VAL_PROBA[member] = predict_fn(member_model, X_val)
                ALL_TEST_PROBA[member] = predict_fn(member_model, X_test)
                del member_model
                cleanup_torch()

        member_val_probas = [ALL_VAL_PROBA[m] for m in hybrid_members]
        member_test_probas = [ALL_TEST_PROBA[m] for m in hybrid_members]

        if hybrid_strategy == "soft_voting":

            hybrid_val_proba = np.mean(member_val_probas, axis=0)
            hybrid_test_proba = np.mean(member_test_probas, axis=0)

        else:  # stacking

            meta_model_path = TRAINED_MODEL_DIR / f"hybrid_{'_'.join(hybrid_members)}_meta_model.pkl"

            if not meta_model_path.exists():
                raise FileNotFoundError(
                    f"{meta_model_path}\n"
                    f"  This must be the EXACT member list (and order) 03_Big_Modeling.ipynb "
                    f"used when it trained the stacking meta-model. Check HYBRID_MEMBERS in "
                    f"the Master Control Cell, or set HYBRID_STRATEGY='soft_voting' if no "
                    f"meta-model was saved."
                )

            meta_model = joblib.load(meta_model_path)

            meta_X_val = np.concatenate(member_val_probas, axis=1)
            meta_X_test = np.concatenate(member_test_probas, axis=1)

            hybrid_val_proba = meta_model.predict_proba(meta_X_val)
            hybrid_test_proba = meta_model.predict_proba(meta_X_test)

        fpr_val, tpr_val, auc_val = compute_roc(y_val, hybrid_val_proba)
        fpr_test, tpr_test, auc_test = compute_roc(y_test, hybrid_test_proba)

        print(f"  Validation AUC : {auc_val:.4f}")
        print(f"  Test AUC       : {auc_test:.4f}")

        out_path = plot_val_test_roc(hybrid_display_name, fpr_val, tpr_val, auc_val,
                                      fpr_test, tpr_test, auc_test)
        print(f"  Saved figure   : {out_path}")

        summary_rows.append({
            "model": hybrid_display_name,
            "status": "ok",
            "val_auc": round(auc_val, 4),
            "test_auc": round(auc_test, 4),
            "generalization_gap": round(auc_val - auc_test, 4),
        })

    except Exception as e:
        print(f"  SKIPPED hybrid: {type(e).__name__}: {e}")
        summary_rows.append({"model": "Hybrid", "status": f"error: {e}"})

    print()

print("Done.")


## Summary Table — Val vs Test AUC, All Models


In [ ]:
# ==============================================================================
# Cell 11 : Summary Table
# ==============================================================================
summary_df = pd.DataFrame(summary_rows)

if "test_auc" in summary_df.columns:
    summary_df = summary_df.sort_values("test_auc", ascending=False, na_position="last")

print("=" * 80)
print(f"ROC AUC Summary -- Validation vs Test ({CONFIG['task_mode']} mode)")
print("=" * 80)
display(summary_df)

summary_path = TABLE_DIR / f"roc_val_test_auc_summary_{CONFIG['task_mode']}.csv"
summary_df.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)


## (Optional) Combined Comparison Figure — All Models, Test Set Only

A bonus overlay figure comparing every model's test-set ROC curve on one
axis. Skip this cell if you only want the per-model val/test figures above.


In [ ]:
# ==============================================================================
# Cell 12 (Optional) : All-Models Test-Set ROC Overlay
# ==============================================================================
fig, ax = plt.subplots(figsize=(8, 7))

for model_key in MODELS_TO_RUN:

    display_name = DISPLAY_NAMES.get(model_key, model_key)

    if model_key not in MODEL_REGISTRY:
        continue

    loader_fn, predict_fn = MODEL_REGISTRY[model_key]

    try:
        model = loader_fn()
        test_proba = predict_fn(model, X_test)
        fpr_test, tpr_test, auc_test = compute_roc(y_test, test_proba)
        ax.plot(fpr_test, tpr_test, lw=2, label=f"{display_name} (AUC = {auc_test:.4f})")
        del model
        cleanup_torch()
    except Exception as e:
        print(f"  SKIPPED {display_name}: {type(e).__name__}: {e}")
        continue

ax.plot([0, 1], [0, 1], linestyle="--", color="grey", lw=1, label="Chance")
curve_kind = "Binary" if CONFIG["task_mode"] == "binary" else "Macro-Average OvR"
ax.set_title(f"All Models — {curve_kind} ROC Curve (Test Set)", fontsize=14)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
out_path = FIGURE_DIR / f"all_models_test_roc_overlay.{CONFIG['figure_format']}"
plt.savefig(out_path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
plt.show()
print("Saved:", out_path)


## Final decision — select by validation AUC

Ranks successfully evaluated models by validation AUC. The held-out test AUC
is displayed only as a final generalization result and is not used to choose the
winner. A large validation-to-test gap is flagged for review.


In [ ]:
# ==============================================================================
# Cell 13 : Final Decision — Select by Validation AUC
# ==============================================================================
ranked_df = summary_df[summary_df.get("status") == "ok"].copy() if "status" in summary_df.columns else summary_df.copy()

print("=" * 80)
print(f"FINAL DECISION -- Selected by Validation AUC ({CONFIG['task_mode']} mode)")
print("=" * 80)

if ranked_df.empty or "val_auc" not in ranked_df.columns:
    print("No models were successfully evaluated. Check missing-file/load warnings.")
else:
    ranked_df = ranked_df.sort_values("val_auc", ascending=False).reset_index(drop=True)
    display(ranked_df[["model", "val_auc", "test_auc", "generalization_gap"]])

    best_row = ranked_df.iloc[0]
    best_model = best_row["model"]
    best_val_auc = float(best_row["val_auc"])
    best_test_auc = float(best_row["test_auc"])
    best_gap = float(best_row["generalization_gap"])
    gap_warning_threshold = 0.02

    print(f"\nSELECTED MODEL : {best_model}")
    print(f"  Validation AUC       : {best_val_auc:.4f}  (selection metric)")
    print(f"  Held-out test AUC    : {best_test_auc:.4f}  (final report only)")
    print(f"  Generalization gap   : {best_gap:.4f}")
    if abs(best_gap) > gap_warning_threshold:
        print(f"  CAUTION: absolute AUC gap exceeds {gap_warning_threshold:.2f}.")

    decision_summary = {
        "task_mode": CONFIG["task_mode"],
        "selection_metric": "validation_auc",
        "selected_model": best_model,
        "validation_auc": round(best_val_auc, 4),
        "held_out_test_auc": round(best_test_auc, 4),
        "generalization_gap": round(best_gap, 4),
    }
    decision_path = TABLE_DIR / f"best_model_decision_{CONFIG['task_mode']}.csv"
    pd.DataFrame([decision_summary]).to_csv(decision_path, index=False)
    print("\nSaved:", decision_path)
